# 1) The AC power flow on a real network — Exercises

**Goals:** count the equations, test the Jacobian against finite differences, see what a simplified Jacobian does to convergence, add generator reactive-power limits, and find how much those limits lower the loadability of the network.

The setup cell holds `case14_data(load_scale, enforce_q_lims)` (the IEEE 14-bus system solved by pandapower, with its internal Y-bus, injections `S`, start voltages `V0`, bus lists and ordering), the lecture's `analytic_jacobian` and `newton_raphson`. Everything is in per unit on 100 MVA unless stated.

**Submit** your notebook for review before opening the solution in `solutions/`.

In [1]:
import logging
import warnings

import numpy as np
import pandas as pd
import pandapower as pp
import pandapower.networks as pn

warnings.filterwarnings("ignore")
logging.getLogger("pandapower").setLevel(logging.ERROR)


def case14_data(load_scale: float = 1.0, enforce_q_lims: bool = False):
    """IEEE 14-bus system with all loads scaled, solved by pandapower. Returns the net and its internal data."""
    net = pn.case14()
    net.load[["p_mw", "q_mvar"]] *= load_scale
    pp.runpp(net, init="flat", numba=False, tolerance_mva=1e-10, enforce_q_lims=enforce_q_lims)
    internal = net._ppc["internal"]
    order = net._pd2ppc_lookups["bus"]
    V0 = np.ones(len(internal["Sbus"]), dtype=complex)
    for table in (net.gen, net.ext_grid):
        V0[order[table.bus.to_numpy()]] = table.vm_pu.to_numpy()
    return net, {"Y": internal["Ybus"].toarray(), "S": internal["Sbus"].copy(), "V0": V0, "ref": internal["ref"],
                 "pv": internal["pv"], "pq": internal["pq"], "order": order, "base": net._ppc["baseMVA"]}


def analytic_jacobian(Y, V, pv, pq):
    """The lecture's polar Jacobian: rows P(pv+pq), Q(pq); columns angle(pv+pq), magnitude(pq)."""
    pvpq = np.r_[pv, pq]
    I = Y @ V
    Vn = V / np.abs(V)
    dS_dVa = 1j * np.diag(V) @ np.conj(np.diag(I) - Y @ np.diag(V))
    dS_dVm = np.diag(V) @ np.conj(Y @ np.diag(Vn)) + np.conj(np.diag(I)) @ np.diag(Vn)
    return np.block([[dS_dVa[np.ix_(pvpq, pvpq)].real, dS_dVm[np.ix_(pvpq, pq)].real],
                     [dS_dVa[np.ix_(pq, pvpq)].imag, dS_dVm[np.ix_(pq, pq)].imag]])


def newton_raphson(Y, S, V0, ref, pv, pq, tol=1e-10, max_iter=30):
    """The lecture's solver: complex voltages and the largest mismatch per iteration."""
    V = V0.astype(complex).copy()
    pvpq = np.r_[pv, pq]
    history = []
    for _ in range(max_iter + 1):
        mismatch = V * np.conj(Y @ V) - S
        F = np.r_[mismatch[pvpq].real, mismatch[pq].imag]
        history.append(np.abs(F).max())
        if history[-1] < tol:
            return V, history
        dx = np.linalg.solve(analytic_jacobian(Y, V, pv, pq), -F)
        Va, Vm = np.angle(V), np.abs(V)
        Va[pvpq] += dx[:len(pvpq)]
        Vm[pq] += dx[len(pvpq):]
        V = Vm * np.exp(1j * Va)
    raise RuntimeError(f"no convergence in {max_iter} iterations")

### Warm-ups

**1. Equations and injections**

(a) Write `count_equations(n_pv, n_pq)` returning (number of equations, unknown angles, unknown voltage magnitudes).
(b) Write `injections(V, Y)` returning the complex power injected at every bus. Check: at pandapower's solution, the injections equal the specified ones in P at PV and PQ buses and in Q at PQ buses.

In [2]:
def count_equations(n_pv: int, n_pq: int) -> tuple[int, int, int]:
    """(equations, unknown angles, unknown magnitudes) of the AC power flow."""
    ### BEGIN SOLUTION
    return n_pv + 2 * n_pq, n_pv + n_pq, n_pq
    ### END SOLUTION


def injections(V: np.ndarray, Y: np.ndarray) -> np.ndarray:
    """Complex power injected at each bus (p.u.)."""
    ### BEGIN SOLUTION
    return V * np.conj(Y @ V)
    ### END SOLUTION

In [3]:
net, d = case14_data()
assert count_equations(len(d["pv"]), len(d["pq"])) == (22, 13, 9)
V_pp = net._ppc["internal"]["V"]                      # pandapower's solution, internal order
S_pp = injections(V_pp, d["Y"])
pvpq = np.r_[d["pv"], d["pq"]]
assert np.abs(S_pp[pvpq].real - d["S"][pvpq].real).max() < 1e-9 and np.abs(S_pp[d["pq"]].imag - d["S"][d["pq"]].imag).max() < 1e-9
print("slack injection (MW, Mvar):", np.round(S_pp[d["ref"][0]] * d["base"], 2))

slack injection (MW, Mvar): (232.39-16.55j)


**2. Is the Jacobian right?**

Write `jacobian_fd(Y, S, V, pv, pq, eps=1e-7)`: the Jacobian by **finite differences** — perturb each angle (PV and PQ buses) and each magnitude (PQ buses) by `eps` and record how the mismatch vector changes. Compare it with `analytic_jacobian` at the flat start and at the solution.

In [4]:
def jacobian_fd(Y, S, V, pv, pq, eps: float = 1e-7) -> np.ndarray:
    """Jacobian of the mismatch [P(pv+pq), Q(pq)] w.r.t. [angle(pv+pq), magnitude(pq)] by forward differences."""
    ### BEGIN SOLUTION
    pvpq = np.r_[pv, pq]

    def mismatch(Va, Vm):
        m = injections(Vm * np.exp(1j * Va), Y) - S
        return np.r_[m[pvpq].real, m[pq].imag]

    Va, Vm = np.angle(V), np.abs(V)
    base = mismatch(Va, Vm)
    columns = []
    for b in pvpq:
        Va2 = Va.copy(); Va2[b] += eps
        columns.append((mismatch(Va2, Vm) - base) / eps)
    for b in pq:
        Vm2 = Vm.copy(); Vm2[b] += eps
        columns.append((mismatch(Va, Vm2) - base) / eps)
    return np.column_stack(columns)
    ### END SOLUTION

In [5]:
for label, V in {"flat start": d["V0"], "solution": V_pp}.items():
    gap = np.abs(jacobian_fd(d["Y"], d["S"], V, d["pv"], d["pq"]) - analytic_jacobian(d["Y"], V, d["pv"], d["pq"])).max()
    assert gap < 1e-4, label
    print(f"{label}: largest difference {gap:.1e}")

flat start: largest difference 3.9e-06
solution: largest difference 4.0e-06


### Core

**3. A simplified Jacobian**

Fast power-flow methods drop the weak couplings between angles and reactive power, and between voltage magnitudes and active power. Write `decoupled_newton(Y, S, V0, ref, pv, pq)`: each iteration solves only the ∂P/∂θ block for the angle update and the ∂Q/∂|V| block for the magnitude update (both taken from `analytic_jacobian`). Compare its iterations and its mismatch history with full Newton–Raphson, and check that it reaches the same solution.

In [6]:
def decoupled_newton(Y, S, V0, ref, pv, pq, tol=1e-10, max_iter=100):
    """Newton-Raphson with only the dP/dangle and dQ/d|V| blocks. Returns V and the mismatch history."""
    ### BEGIN SOLUTION
    V = V0.astype(complex).copy()
    pvpq = np.r_[pv, pq]
    n = len(pvpq)
    history = []
    for _ in range(max_iter + 1):
        m = injections(V, Y) - S
        dP, dQ = m[pvpq].real, m[pq].imag
        history.append(max(np.abs(dP).max(), np.abs(dQ).max()))
        if history[-1] < tol:
            return V, history
        J = analytic_jacobian(Y, V, pv, pq)
        Va, Vm = np.angle(V), np.abs(V)
        Va[pvpq] += np.linalg.solve(J[:n, :n], -dP)
        Vm[pq] += np.linalg.solve(J[n:, n:], -dQ)
        V = Vm * np.exp(1j * Va)
    raise RuntimeError("no convergence")
    ### END SOLUTION

In [7]:
V_full, h_full = newton_raphson(d["Y"], d["S"], d["V0"], d["ref"], d["pv"], d["pq"])
V_dec, h_dec = decoupled_newton(d["Y"], d["S"], d["V0"], d["ref"], d["pv"], d["pq"])
assert np.abs(V_dec - V_full).max() < 1e-8, "same solution"
assert len(h_dec) > len(h_full), "the simplified Jacobian needs more iterations"
ratios = np.array(h_dec[2:]) / np.array(h_dec[1:-1])
print(f"full Newton: {len(h_full) - 1} iterations; decoupled: {len(h_dec) - 1} iterations; "
      f"decoupled mismatch shrinks by a factor of about {np.median(ratios):.2f} per step")
print("full:      ", "  ".join(f"{h:.0e}" for h in h_full))
print("decoupled: ", "  ".join(f"{h:.0e}" for h in h_dec[:8]), " ...", f"{h_dec[-1]:.0e} after {len(h_dec) - 1} iterations")

full Newton: 4 iterations; decoupled: 88 iterations; decoupled mismatch shrinks by a factor of about 0.80 per step
full:       9e-01  1e-01  7e-04  6e-08  1e-14
decoupled:  9e-01  5e-01  1e-01  9e-02  2e-02  2e-02  8e-03  6e-03  ... 9e-11 after 88 iterations


**4. Generator reactive-power limits**

At 1.3 times the base load some generators would have to produce more reactive power than they can. Write `solve_with_q_limits(load_scale)`: solve; compute each generator's reactive output (MVAr) — the bus's reactive injection plus its local load; if a generator is outside its limits, fix its output at the limit and turn its bus from PV into PQ; re-solve until no limit is violated. Return the voltages (in pandapower bus order), the generators that hit a limit and the number of rounds. Check against pandapower with `enforce_q_lims=True`.

In [8]:
def solve_with_q_limits(load_scale: float) -> dict:
    """Own power flow with PV-to-PQ switching at generator reactive limits."""
    ### BEGIN SOLUTION
    net, d = case14_data(load_scale)
    S, pv, pq = d["S"].copy(), list(d["pv"]), list(d["pq"])
    gen_bus = d["order"][net.gen.bus.to_numpy()]
    q_max, q_min = net.gen.max_q_mvar.to_numpy(), net.gen.min_q_mvar.to_numpy()
    at_limit = {}
    for rounds in range(1, 10):
        V, _ = newton_raphson(d["Y"], S, d["V0"], d["ref"], np.array(pv, int), np.array(pq, int))
        q_gen = (injections(V, d["Y"]).imag - S.imag)[gen_bus] * d["base"]      # S holds -load Q at PV buses
        violated = [k for k, b in enumerate(gen_bus) if b in pv and not (q_min[k] - 1e-6 <= q_gen[k] <= q_max[k] + 1e-6)]
        if not violated:
            break
        for k in violated:
            limit = q_max[k] if q_gen[k] > q_max[k] else q_min[k]
            S[gen_bus[k]] += 1j * limit / d["base"]
            pv.remove(gen_bus[k]); pq.append(gen_bus[k])
            at_limit[int(net.gen.bus.iloc[k]) + 1] = limit
    return {"vm_pu": pd.Series(np.abs(V)[d["order"][net.bus.index]], index=net.bus.index + 1),
            "at_limit_mvar": at_limit, "rounds": rounds}
    ### END SOLUTION

In [9]:
own = solve_with_q_limits(1.3)
reference, _ = case14_data(1.3, enforce_q_lims=True)
assert np.abs(own["vm_pu"].to_numpy() - reference.res_bus.vm_pu.to_numpy()).max() < 1e-8
no_limits, _ = case14_data(1.3)
print("generators at their limit (bus: Mvar):", own["at_limit_mvar"], "after", own["rounds"], "rounds")
pd.DataFrame({"without limits": no_limits.res_bus.vm_pu.to_numpy(), "with limits": own["vm_pu"].to_numpy()},
             index=own["vm_pu"].index).T.round(3)

generators at their limit (bus: Mvar): {2: np.float64(50.0), 3: np.float64(40.0), 6: np.float64(24.0), 8: np.float64(24.0)} after 3 rounds


                  1      2      3      4      5     6      7      8      9   \
without limits  1.06  1.045  1.010  1.008  1.011  1.07  1.052  1.090  1.042   
with limits     1.06  1.026  0.981  0.986  0.991  1.04  1.027  1.067  1.015   

                   10     11     12     13     14  
without limits  1.037  1.049  1.050  1.043  1.018  
with limits     1.009  1.020  1.019  1.012  0.989  

**5. Interpretation**

What does the decoupled method's mismatch history tell you about its convergence rate, and why does it still reach the exact solution? What happened to the voltages when the generators reached their reactive limits, and why?

<!-- BEGIN SOLUTION -->
The decoupled method converges **linearly**: its mismatch shrinks by a roughly constant factor of about 0.80 per iteration, so it needs 88 iterations where full Newton needs 4 — no doubling of correct digits, because it no longer uses the exact derivative. It still reaches the **exact** solution (to 10⁻⁸), because the equations it drives to zero are unchanged: a simplified Jacobian changes only the path to the solution, not the solution itself. How much speed is lost depends on how strong the neglected couplings are. In this network they are not small — many branches have a resistance of 0.3–0.5 times their reactance, so active power also responds to voltage magnitude. Alternating the two half-steps (update the angles, recompute the reactive mismatch, then update the magnitudes) roughly halves the count, to 46 iterations on this network. Fast decoupled methods (lesson 2) go further and replace the blocks with constant matrices that are factorised once — each iteration becomes much cheaper, which is the real point of decoupling on large networks.

At 1.3 times the base load all four generators (buses 2, 3, 6 and 8) reach their upper reactive limits (50, 40, 24 and 24 Mvar) within three rounds of switching. A generator at its limit can no longer hold its voltage set point: its bus becomes a PQ bus with fixed Q, and the voltage drops there and everywhere it supported — bus 2 from 1.045 to 1.026 p.u., bus 14 from 1.018 to 0.989 p.u. The network has lost its voltage control and now relies on the slack alone.
<!-- END SOLUTION -->

### Challenge

**6. How much do reactive limits cost in loadability?**

Find the largest load scale (to 0.01) at which the power flow still converges — by bisection — once **without** reactive limits (scaling only the loads, generators' active power unchanged) and once **with** them (`solve_with_q_limits`). Treat a failed solve as "beyond the limit". Write `loadability(with_limits)`.

In [10]:
def loadability(with_limits: bool, low: float = 1.0, high: float = 6.0, tol: float = 0.01) -> float:
    """Largest load scale that still solves, by bisection."""
    ### BEGIN SOLUTION
    net, d = case14_data(1.0)
    S_load = np.zeros(len(d["S"]), dtype=complex)                      # load at every bus, internal order (p.u.)
    np.add.at(S_load, d["order"][net.load.bus.to_numpy()],
              (net.load.p_mw.to_numpy() + 1j * net.load.q_mvar.to_numpy()) / d["base"])
    S_gen = d["S"] + S_load                                            # what the generators inject

    def solves(scale):
        try:
            if with_limits:
                solve_with_q_limits(scale)
            else:                                                       # the same load scaling, generators unchanged
                newton_raphson(d["Y"], S_gen - scale * S_load, d["V0"], d["ref"], d["pv"], d["pq"])
            return True
        except Exception:
            return False
    while high - low > tol:
        mid = (low + high) / 2
        low, high = (mid, high) if solves(mid) else (low, mid)
    return low
    ### END SOLUTION

In [11]:
without = loadability(False)
with_q = loadability(True)
assert with_q < without, "reactive limits lower the loadability"
print(f"loadability without reactive limits: {without:.2f} x base load; with limits: {with_q:.2f} x")

loadability without reactive limits: 4.00 x base load; with limits: 1.75 x


**7. Interpretation**

How large is the difference, and what does it say about studies that ignore generator reactive limits? Which kind of real-world equipment would raise the limit again?

<!-- BEGIN SOLUTION -->
Reactive limits more than halve the loadability: from 4.00 times the base load without limits to 1.75 times with them (pandapower's own limit enforcement agrees — it still converges at 1.76 and fails at 1.78). A study that ignores generator reactive limits would overstate how much load this network can carry by a factor of more than two, and would miss that the limit is set by **reactive power and voltage**, not by active power or line ratings. Reactive power cannot be transported far — it is consumed in the line reactances — so the fix has to be local: shunt capacitor banks, static VAR compensators or STATCOMs, synchronous condensers, or generators with a larger reactive capability. Lesson 7 returns to voltage and reactive power in detail.
<!-- END SOLUTION -->